# Exercise 2 - Inverse Perturbation (Two States)

Objective: predict perturbation from two consecutive states with `dataset_mode="inverse_perturbation_two_states"`.

This mode uses `[X_t, X_{t+1}] -> P_t`, so the MLP shape is `(2 * expression_dim, hidden_d, perturbation_dim)` (here `(100, hidden_d, 8)`).


In [ ]:
from pathlib import Path
import sys
import pandas as pd

sys.path.append(str(Path.cwd() / 'src'))

from perturbation_pipeline import (
    evaluate_predictions_by_timepoint,
    load_experiment_data,
    merge_train_val_splits,
    split_dataset,
    test,
    train_with_model_selection,
)


In [ ]:
DATA_DIR = Path('simulated_data_for_interview_exercise')
adata = load_experiment_data(DATA_DIR)

print('adata shape:', adata.shape)
print('expression dim:', adata.n_vars)
print('perturbation dim:', adata.obsm['perturbation'].shape[1])
print('timepoints:', sorted(adata.obs['round'].unique().tolist()))


In [ ]:
splits = split_dataset(
    adata=adata,
    test_timepoints=(9, 10),
    val_fraction=0.2,
    random_state=42,
    dataset_mode="inverse_perturbation_two_states",
)

for split_name in ['train', 'val', 'test']:
    x_split, y_split = splits[split_name]
    print(split_name, 'X:', x_split.shape, 'y:', y_split.shape)

input_dim = splits['train'][0].shape[1]
output_dim = splits['train'][1].shape[1]
print('MLP shape for this mode:', (input_dim, 'hidden_d', output_dim))


In [ ]:
# Keep this example simple: choose among a small set of models
train_pool = merge_train_val_splits(splits)

selected = train_with_model_selection(
    train_data=train_pool,
    candidate_models=('mlp', 'linear_regression'),
    n_splits=5,
    random_state=42,
    mlp_hidden_dim=128,
    mlp_max_iter=300,
)

print('Best model:', selected['best_model_name'])

summary_rows = []
for model_name, result in selected['cv_results'].items():
    row = {'model': model_name}
    row.update({f'mean_{k}': v for k, v in result['mean_metrics'].items()})
    row.update({f'std_{k}': v for k, v in result['std_metrics'].items()})
    summary_rows.append(row)

pd.DataFrame(summary_rows).sort_values('mean_rmse').reset_index(drop=True)


In [ ]:
test_output = test(selected['model'], splits['test'])
print('Test metrics:', test_output['metrics'])

per_tp_metrics = evaluate_predictions_by_timepoint(
    prediction=test_output['prediction'],
    ground_truth=splits['test'][1],
    timepoints=splits['timepoints']['test'],
)
per_tp_metrics
